# Stretch: more joins

For when the live exercise is done and there is time left. These get
harder as you go, so nobody is expected to finish them all.

The habits from the live exercise still hold: alias every table, and after
every join, ask yourself how many rows you expect and check that you got
them.

Two of them need something new, and the notebook tells you what it is when
you get there.

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")

## Stretch 1: who did I not play in 2025?

Every artist with their number of plays **in 2025**, including the ones
with 0, fewest first.

Write the obvious version first: a `LEFT JOIN` and a `WHERE` for the year.
Before you run it, how many rows should come back? Then run it and look at
the row count.

In [ ]:
run("""
""")

If the count is not what you expected, the fix is to move the date
condition out of the `WHERE` and into the `ON`, with an `AND`. Try it, and
check the count again.

In [ ]:
run("""
""")

Now list **only** the artists with no 2025 plays at all. Keep the same
join, and add `WHERE p.play_id IS NULL`.

In [ ]:
run("""
""")

**Why** did the first `WHERE` break the left join, when this `WHERE` does
not? Both run after the join. Think about which rows each one keeps.

*Your answer here.*

## Stretch 2: artists from the same country

List every pair of artists that come from the same country, like
`Germany | Glass Tram | Hertz Garden`.

Both halves of the pair are in `artists`, so you join `artists` to
**itself**. That is allowed, as long as each copy gets its own alias:

```sql
FROM artists a1
JOIN artists a2 ON a2.country = a1.country
```

Run that much first and count the rows. Then fix it so each pair appears
exactly once, and Glass Tram is never paired with Glass Tram. (Hint: each
artist has a different `artist_id`, and one of `<>` or `>` is better than
the other.)

In [ ]:
run("""
""", limit=25)

You should get 22 pairs. Can you check that by hand, from how many artists
each country has?

**Harder:** only the pairs that share a genre as well.

In [ ]:
run("""
""")

## Stretch 3: 2024 against 2025

For every artist: minutes in 2024, minutes in 2025, and the change. Biggest
riser at the top.

Build it in steps, with a CTE for each:

1. `minutes_2024`: one row per artist, their 2024 minutes
2. `minutes_2025`: the same for 2025
3. `compared`: every artist, with both numbers side by side

CTEs are separated by commas, and a later one can use the earlier ones.

Two things to watch:

* Some artists have minutes in 2024 and none in 2025. Make sure they do not
  disappear. Which table should `compared` start from?
* What is `NULL` minus a number? `COALESCE` from the live exercise will
  help.

In [ ]:
run("""
""", limit=40)

**Harder:** which artists did I stop listening to? Played in 2024, not at
all in 2025. With `compared` already built, this should be a small change
at the bottom.

In [ ]:
run("""
""")

## Stretch 4: spot the bug

Someone wants the number of awards and the number of plays for each genre,
and writes this. Before you run it, use what you know: there are 33 awards
in the whole table. What should the awards column add up to?

In [ ]:
run("""
SELECT a.genre,
       COUNT(w.award_id) AS awards,
       COUNT(p.play_id)  AS plays
FROM artists a
LEFT JOIN awards w USING (artist_id)
LEFT JOIN plays  p USING (artist_id)
GROUP BY a.genre
ORDER BY a.genre
""")

1. What went wrong, and why? Fjord & Flint (Folk, 3 awards, 237 plays) is
   a good artist to work it out with.
2. Some genres have the right play count anyway. Which ones, and why
   them?

*Your answer here.*

Now fix it by changing only the two `COUNT`s, using `DISTINCT`.

In [ ]:
run("""
""")

In notebook 01, `SUM(DISTINCT minutes_played)` was a trap that gave the
wrong answer. **Why is `COUNT(DISTINCT w.award_id)` a correct fix here?**
What is different about `award_id` compared with `minutes_played`?

*Your answer here.*

**Harder:** add total minutes per genre as a third column. `DISTINCT` will
not save you this time, so you will need CTEs.

In [ ]:
run("""
""")

## Stretch 5: the top artist in every genre

Who do I listen to most (by minutes) in each genre? One row per genre.

`GROUP BY` cannot answer this. It gives you one number per genre, not the
name of who won. You need something new.

**New tool: window functions.** A window function calculates something
across a set of rows *without* squashing them into one row the way
`GROUP BY` does. Every row stays, and gets a new column. Run this:

In [ ]:
run("""
WITH per_genre AS (
    SELECT a.genre, COUNT(*) AS plays
    FROM plays p
    JOIN artists a USING (artist_id)
    GROUP BY a.genre
)
SELECT genre, plays,
       RANK() OVER (ORDER BY plays DESC) AS position
FROM per_genre
""")

`RANK() OVER (ORDER BY plays DESC)` numbers the rows 1, 2, 3, in that
order. Add `PARTITION BY` and the numbering restarts for each group:

```sql
RANK() OVER (PARTITION BY a.genre ORDER BY s.minutes DESC)
```

Now the question. Build it in steps:

1. a CTE with minutes per artist (aggregate first, as always)
2. a CTE that joins on `artists` and ranks each artist within their genre
3. a final `SELECT` that keeps only position 1

In [ ]:
run("""
""")

**Why** does the "keep only position 1" filter need its own step? Try
putting `WHERE position = 1` in the same `SELECT` as the `RANK()` and read
the error. The execution order from session 6 explains it.

*Your answer here.*

**Harder:** the top two per genre.

In [ ]:
run("""
""", limit=16)

## Stretch 6: when was I halfway?

Over the two years I listened to 8980.4 minutes. In which month did I pass
half of that?

First, minutes per month with a **running total**: each month's minutes
plus everything before it. A window function does that too:

```sql
SUM(minutes) OVER (ORDER BY month)
```

`strftime('%Y-%m', played_at)` turns a date into a month like `2024-03`.
Put the monthly totals in a CTE, then add the running total on top.

In [ ]:
run("""
""", limit=24)

Check the last row against the 8980.4 above. If it is slightly off, look
at where you rounded.

Now the month itself. One more CTE for the running totals, one for half
the total, then find the first month whose running total reached it.

In [ ]:
run("""
""")

**Harder:** make the running total start again from zero each year.
(Stretch 5 had the word you need.)

In [ ]:
run("""
""", limit=24)

In [ ]:
con.close()